# Bluestock Fintech — Mutual Fund Analytics Capstone
## Day 3: Exploratory Data Analysis (EDA)

Covers NAV trends, AUM growth, SIP inflows, category heatmaps, investor demographics,
geographic distribution, folio growth, correlation matrix, sector allocation, and key findings.


In [ ]:
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)

# Auto-detect data folder (Kaggle input or local)
def find_data_dir():
    kaggle_matches = glob.glob("/kaggle/input/*/01_fund_master.csv")
    if kaggle_matches:
        return os.path.dirname(kaggle_matches[0])
    if os.path.exists("01_fund_master.csv"):
        return "."
    raise FileNotFoundError("Could not find 01_fund_master.csv — check your Kaggle input dataset.")

DATA_DIR = find_data_dir()
print(f"Reading CSVs from: {DATA_DIR}")

In [ ]:
# Load all datasets
fund_master   = pd.read_csv(f"{DATA_DIR}/01_fund_master.csv")
nav_history   = pd.read_csv(f"{DATA_DIR}/02_nav_history.csv", parse_dates=["date"])
aum           = pd.read_csv(f"{DATA_DIR}/03_aum_by_fund_house.csv", parse_dates=["date"])
sip_inflows   = pd.read_csv(f"{DATA_DIR}/04_monthly_sip_inflows.csv")
category_inf  = pd.read_csv(f"{DATA_DIR}/05_category_inflows.csv")
folio_count   = pd.read_csv(f"{DATA_DIR}/06_industry_folio_count.csv")
performance   = pd.read_csv(f"{DATA_DIR}/07_scheme_performance.csv")
transactions  = pd.read_csv(f"{DATA_DIR}/08_investor_transactions.csv", parse_dates=["transaction_date"])
holdings      = pd.read_csv(f"{DATA_DIR}/09_portfolio_holdings.csv")
benchmarks    = pd.read_csv(f"{DATA_DIR}/10_benchmark_indices.csv", parse_dates=["date"])

print("All 10 datasets loaded.")
print(f"Funds: {len(fund_master)} | NAV rows: {len(nav_history)} | Transactions: {len(transactions)}")

## Task 1 — NAV Trend Analysis
Daily NAV for all 40 schemes, 2022–2026.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

for code_, grp in nav_history.groupby("amfi_code"):
    grp_sorted = grp.sort_values("date")
    ax.plot(grp_sorted["date"], grp_sorted["nav"], linewidth=0.7, alpha=0.6)

ax.set_title("NAV Trend — All 40 Schemes (2022–2026)", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("NAV (Rs.)")
plt.tight_layout()
plt.savefig("chart_01_nav_trend.png", dpi=120)
plt.show()

## Task 2 — AUM Growth by Fund House
Grouped bar chart, AUM by fund house per year.

In [ ]:
aum["year"] = aum["date"].dt.year
aum_yearly = aum.groupby(["year", "fund_house"])["aum_crore"].mean().reset_index()

plt.figure(figsize=(14, 7))
sns.barplot(data=aum_yearly, x="fund_house", y="aum_crore", hue="year")
plt.title("Average AUM by Fund House per Year", fontsize=14, fontweight="bold")
plt.xlabel("Fund House")
plt.ylabel("AUM (Rs. Crore)")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Year")
plt.tight_layout()
plt.savefig("chart_02_aum_growth.png", dpi=120)
plt.show()

## Task 3 — SIP Inflow Trend
Monthly SIP inflow, Jan 2022 – Dec 2025.

In [ ]:
fig = px.line(sip_inflows, x="month", y="sip_inflow_crore",
              title="Monthly SIP Inflow (Rs. Crore)",
              labels={"sip_inflow_crore": "SIP Inflow (Rs. Cr)", "month": "Month"})
fig.update_xaxes(tickangle=45)
fig.show()

## Task 4 — Category-wise Inflow Heatmap
Months × categories, net inflow as colour intensity.

In [ ]:
pivot = category_inf.pivot(index="category", columns="month", values="net_inflow_crore")

plt.figure(figsize=(16, 6))
sns.heatmap(pivot, cmap="RdYlGn", center=0, annot=False, cbar_kws={'label': 'Net Inflow (Rs. Cr)'})
plt.title("Category-wise Net Inflows Heatmap", fontsize=14, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Category")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("chart_04_category_heatmap.png", dpi=120)
plt.show()

## Task 5 — Investor Demographics
Age group distribution + SIP amount by age group.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

age_dist = transactions["age_group"].value_counts()
axes[0].pie(age_dist.values, labels=age_dist.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title("Investor Age Group Distribution", fontsize=13, fontweight="bold")

sip_only = transactions[transactions["transaction_type"] == "SIP"]
sns.boxplot(data=sip_only, x="age_group", y="amount_inr", ax=axes[1],
            order=sorted(sip_only["age_group"].unique()))
axes[1].set_title("SIP Amount Distribution by Age Group", fontsize=13, fontweight="bold")
axes[1].set_ylabel("SIP Amount (Rs.)")
axes[1].set_xlabel("Age Group")

plt.tight_layout()
plt.savefig("chart_05_demographics.png", dpi=120)
plt.show()

## Task 6 — Geographic Distribution
SIP amount by state + T30 vs B30 split.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

state_sip = transactions[transactions["transaction_type"] == "SIP"].groupby("state")["amount_inr"].sum().sort_values()
axes[0].barh(state_sip.index, state_sip.values / 1e7, color="steelblue")
axes[0].set_title("Total SIP Amount by State", fontsize=13, fontweight="bold")
axes[0].set_xlabel("SIP Amount (Rs. Crore)")

tier_dist = transactions["city_tier"].value_counts()
axes[1].pie(tier_dist.values, labels=tier_dist.index, autopct='%1.1f%%',
            colors=["#4C72B0", "#DD8452"], startangle=90)
axes[1].set_title("T30 vs B30 Transaction Split", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig("chart_06_geo_distribution.png", dpi=120)
plt.show()

## Task 7 — Folio Count Growth
Total MF folios, Jan 2022 – Dec 2025.

In [ ]:
fig = px.line(folio_count, x="month", y="total_folios_crore",
              title="Total Mutual Fund Folio Count Growth (Rs. Crore Folios)",
              labels={"total_folios_crore": "Folios (Crore)", "month": "Quarter"},
              markers=True)
fig.show()

## Task 8 — Correlation Matrix
Pairwise correlation of NAV daily returns across 10 selected funds.

In [ ]:
# Compute daily returns
nav_sorted = nav_history.sort_values(["amfi_code", "date"])
nav_sorted["daily_return"] = nav_sorted.groupby("amfi_code")["nav"].pct_change()

# Pick 10 funds for readability
sample_codes = fund_master["amfi_code"].unique()[:10]
pivot_returns = nav_sorted[nav_sorted["amfi_code"].isin(sample_codes)].pivot(
    index="date", columns="amfi_code", values="daily_return"
)

corr = pivot_returns.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("NAV Return Correlation Matrix (10 Sample Funds)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("chart_08_correlation_matrix.png", dpi=120)
plt.show()

## Task 9 — Top Holdings Sector Distribution
Donut chart of sector weights across all equity fund portfolios.

In [ ]:
sector_weights = holdings.groupby("sector")["weight_pct"].sum().sort_values(ascending=False)

plt.figure(figsize=(9, 9))
colors = sns.color_palette("Set3", len(sector_weights))
wedges, texts, autotexts = plt.pie(sector_weights.values, labels=sector_weights.index,
                                     autopct='%1.1f%%', startangle=90, colors=colors,
                                     wedgeprops=dict(width=0.4))
plt.title("Sector Allocation Across Equity Holdings", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("chart_09_sector_allocation.png", dpi=120)
plt.show()

## Task 10 — Key EDA Findings

In [ ]:
# Compute a few quick stats to back up findings
top_fund_house = aum.groupby("fund_house")["aum_crore"].mean().idxmax()
top_sip_month = sip_inflows.loc[sip_inflows["sip_inflow_crore"].idxmax(), "month"]
top_sector = sector_weights.idxmax()
top_state = state_sip.idxmax()
b30_pct = (tier_dist.get("B30", 0) / tier_dist.sum()) * 100

print(f"Top fund house by average AUM   : {top_fund_house}")
print(f"Peak SIP inflow month           : {top_sip_month}")
print(f"Most concentrated sector         : {top_sector}")
print(f"Highest SIP contributing state  : {top_state}")
print(f"B30 share of transactions        : {b30_pct:.1f}%")

**Summary of findings:**

1. NAV trends across all 40 schemes show broad upward momentum from 2022 to 2026, with visible volatility clusters around known market correction periods.
2. AUM growth is concentrated among a handful of large fund houses, consistent with industry consolidation trends.
3. SIP inflows show a steady upward trajectory, peaking toward the most recent months in the dataset.
4. Category-wise inflows vary significantly by month, with certain categories (e.g. Small Cap, ELSS) showing more volatile inflow/outflow patterns than others.
5. The 26–35 and 36–45 age groups contribute the largest share of SIP volume.
6. Geographic contribution remains skewed toward established T30 cities, though B30 representation is non-trivial.
7. Folio count has grown consistently across the observed period, reflecting expanding retail participation.
8. Fund return correlations are generally positive across similar-category funds, with lower correlation for debt/liquid-style schemes.
9. Sector allocation in equity holdings is concentrated in a few dominant sectors (typically Banking/Financial Services and IT).
10. Overall, the data reflects a maturing retail mutual fund market with broadening participation but continued concentration at the fund-house and sector level.

*Note: exact rankings and percentages above are computed directly from the dataset — refer to the printed output for this run's specific values.*
